# Scanned Images Classifier (PyTorch) - Adaptation locale

Source demande: [Kaggle Notebook](https://www.kaggle.com/code/killa92/acc-0-85-scanned-images-classifier-pytorch/notebook)

## Note importante
- L acces direct au notebook Kaggle a ete bloque dans cet environnement (page dynamique + anti-forgery).
- Ce notebook est une adaptation locale executable, avec la meme finalite: entrainer un classifieur d images scannees avec PyTorch.
- Un fallback sur dataset synthetique permet un test integral sans dependance externe.


In [ ]:
from pathlib import Path
import json
import sys

if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent.resolve()
else:
    PROJECT_ROOT = Path.cwd().resolve()

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from scanned_images_pipeline import PipelineConfig, run_training_pipeline

PROJECT_ROOT


## Configuration du run
- `data_root`: dossier dataset local (`data/scanned_images`)
- `use_synthetic_if_missing=True`: genere un mini dataset synthetique si `train/` + `val/` manquent
- `use_pretrained=False`: evite le telechargement de poids ImageNet en mode offline


In [ ]:
DATA_ROOT = PROJECT_ROOT / "data" / "scanned_images"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

config = PipelineConfig(
    project_root=PROJECT_ROOT,
    data_root=DATA_ROOT,
    artifacts_dir=ARTIFACTS_DIR,
    batch_size=8,
    img_size=128,
    epochs=1,
    learning_rate=1e-3,
    num_workers=0,
    seed=42,
    use_pretrained=False,
    use_synthetic_if_missing=True,
)

config


In [ ]:
summary = run_training_pipeline(config)

print("=== RUN SUMMARY ===")
print(json.dumps({k: v for k, v in summary.items() if k != "classification_report"}, indent=2))
print("\n=== CLASSIFICATION REPORT ===")
print(summary["classification_report"])


## Passage en entrainement reel
Quand ton dataset est pret, passe en mode reel:

1. Place les images dans:
   - `data/scanned_images/train/<classe>/...`
   - `data/scanned_images/val/<classe>/...`
2. Active les poids pre-entraines (si internet dispo) et desactive le fallback synthetique.


In [ ]:
# Exemple de configuration reelle (decommenter pour lancer):
# real_config = PipelineConfig(
#     project_root=PROJECT_ROOT,
#     data_root=DATA_ROOT,
#     artifacts_dir=ARTIFACTS_DIR,
#     batch_size=16,
#     img_size=224,
#     epochs=8,
#     learning_rate=1e-3,
#     num_workers=0,
#     seed=42,
#     use_pretrained=True,
#     use_synthetic_if_missing=False,
# )
# real_summary = run_training_pipeline(real_config)
# print(json.dumps({k: v for k, v in real_summary.items() if k != "classification_report"}, indent=2))


## Verification de l artifact modele
Le pipeline sauvegarde automatiquement:
- `artifacts/scanned_images_resnet18.pt`


In [ ]:
import torch

checkpoint_path = Path(summary["model_path"])
print("Checkpoint exists:", checkpoint_path.exists())
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    print("Classes:", checkpoint.get("class_names"))
